# 恒瑞医药 A股 vs 港股 数据获取与对比分析

本 Notebook 展示完整流程：从数据获取 → 数据处理 → 可视化对比。

- **A股**: 600276.SH（上海交易所，数据来源: Tushare Pro）
- **港股**: 01276.HK（香港交易所，数据来源: westock-data）
- **时间**: 2025-07-04 ~ 2026-07-03（近一年）


In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mplfinance as mpf

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("Libraries loaded successfully")


In [ ]:
# 加载 A股数据
with open('hengrui_tushare_600276.json', 'r', encoding='utf-8') as f:
    a_raw = json.load(f)

df_a = pd.DataFrame(a_raw)
df_a['trade_date'] = pd.to_datetime(df_a['trade_date'])
df_a = df_a.sort_values('trade_date').reset_index(drop=True)
df_a = df_a.rename(columns={'vol': 'volume'})

print(f"A股: {len(df_a)} records")
print(f"Range: {df_a['trade_date'].min().date()} ~ {df_a['trade_date'].max().date()}")
df_a.head()


In [ ]:
# 加载港股数据
with open('hengrui_hk_01276.json', 'r', encoding='utf-8') as f:
    h_raw = json.load(f)

df_h = pd.DataFrame(h_raw)
df_h['trade_date'] = pd.to_datetime(df_h['trade_date'])
df_h = df_h.sort_values('trade_date').reset_index(drop=True)

print(f"港股: {len(df_h)} records")
print(f"Range: {df_h['trade_date'].min().date()} ~ {df_h['trade_date'].max().date()}")
df_h.head()


In [ ]:
# 基本统计信息
def show_stats(df, name):
    close = df['close']
    ret = (close.iloc[-1] / close.iloc[0] - 1) * 100
    print(f"=== {name} ===")
    print(f"  记录数: {len(df)}")
    print(f"  起始收盘: {close.iloc[0]:.2f}")
    print(f"  最新收盘: {close.iloc[-1]:.2f}")
    print(f"  区间涨跌幅: {ret:.2f}%")
    print(f"  最高价: {df['high'].max():.2f}")
    print(f"  最低价: {df['low'].min():.2f}")
    print(f"  均价: {close.mean():.2f}")
    print()

show_stats(df_a, "A股 (600276.SH)")
show_stats(df_h, "港股 (01276.HK)")


In [ ]:
# 准备 mplfinance 格式数据
def prep_mpf(df):
    mdf = df[['trade_date','open','high','low','close','volume']].copy()
    mdf = mdf.rename(columns={
        'trade_date':'Date','open':'Open','high':'High',
        'low':'Low','close':'Close','volume':'Volume'
    }).set_index('Date')
    mdf.index = pd.DatetimeIndex(mdf.index)
    return mdf

mpf_a = prep_mpf(df_a)
mpf_h = prep_mpf(df_h)
print(f"A股 OHLCV: {mpf_a.shape}")
print(f"港股 OHLCV: {mpf_h.shape}")


In [ ]:
# A股 K线图
fig, axes = mpf.plot(
    mpf_a, type='candle', volume=True, mav=(5,10,20),
    style='charles', figsize=(14,8), returnfig=True,
    title='恒瑞医药 A股 (600276.SH) K线图\nMA5/MA10/MA20',
    ylabel='Price (CNY)', ylabel_lower='Volume'
)
plt.show()


In [ ]:
# 港股 K线图
fig, axes = mpf.plot(
    mpf_h, type='candle', volume=True, mav=(5,10,20),
    style='yahoo', figsize=(14,8), returnfig=True,
    title='恒瑞医药 港股 (01276.HK) K线图\nMA5/MA10/MA20',
    ylabel='Price (HKD)', ylabel_lower='Volume'
)
plt.show()


In [ ]:
# 归一化价格走势对比
merged = pd.merge(
    df_a[['trade_date','close']].rename(columns={'close':'A_close'}),
    df_h[['trade_date','close']].rename(columns={'close':'H_close'}),
    on='trade_date', how='outer'
).sort_values('trade_date').fillna(method='ffill')

merged['A_idx'] = merged['A_close'] / merged['A_close'].iloc[0] * 100
merged['H_idx'] = merged['H_close'] / merged['H_close'].iloc[0] * 100

fig, ax = plt.subplots(figsize=(14,6))
ax.plot(merged['trade_date'], merged['A_idx'], label='A股', color='#ef232a', lw=2)
ax.plot(merged['trade_date'], merged['H_idx'], label='港股', color='#f5a623', lw=2)
ax.axhline(100, color='#555', ls='--', lw=0.8)
ax.set_title('价格走势对比 (起始日=100)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Normalized Price')
ax.legend(); ax.grid(alpha=0.3)
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

a_chg = merged['A_idx'].iloc[-1] - 100
h_chg = merged['H_idx'].iloc[-1] - 100
print(f"A股: {a_chg:+.2f}%")
print(f"港股: {h_chg:+.2f}%")


In [ ]:
# 成交量对比
vol_m = pd.merge(
    df_a[['trade_date','volume']].rename(columns={'volume':'A_vol'}),
    df_h[['trade_date','volume']].rename(columns={'volume':'H_vol'}),
    on='trade_date', how='inner'
).sort_values('trade_date')

fig, ax1 = plt.subplots(figsize=(14,6))
ax1.bar(vol_m['trade_date'], vol_m['A_vol']/10000,
        width=0.8, color='rgba(239,35,42,0.5)', label='A股 成交量(万手)')
ax1.set_ylabel('A股 (万手)', color='#ef232a')
ax1.tick_params(axis='y', labelcolor='#ef232a')

ax2 = ax1.twinx()
ax2.bar(vol_m['trade_date'], vol_m['H_vol']/10000,
        width=0.8, color='rgba(245,166,35,0.3)', label='港股 成交量(万股)')
ax2.set_ylabel('港股 (万股)', color='#f5a623')
ax2.tick_params(axis='y', labelcolor='#f5a623')

fig.autofmt_xdate()
ax1.set_title('成交量对比', fontsize=14, fontweight='bold')
ax1.grid(alpha=0.3)
l1, la = ax1.get_legend_handles_labels()
l2, lb = ax2.get_legend_handles_labels()
ax1.legend(l1+l2, la+lb, loc='upper left')
plt.tight_layout(); plt.show()

print(f"A股日均: {df_a['volume'].mean()/10000:.0f}万手")
print(f"港股日均: {df_h['volume'].mean()/10000:.0f}万股")


In [ ]:
# 波动率分析
df_a['ret'] = df_a['close'].pct_change()
df_h['ret'] = df_h['close'].pct_change()
df_a['volatility'] = df_a['ret'].rolling(20).std() * np.sqrt(252) * 100
df_h['volatility'] = df_h['ret'].rolling(20).std() * np.sqrt(252) * 100

fig, ax = plt.subplots(figsize=(14,6))
ax.plot(df_a['trade_date'], df_a['volatility'], label='A股', color='#ef232a', lw=1.5)
ax.plot(df_h['trade_date'], df_h['volatility'], label='港股', color='#f5a623', lw=1.5)
ax.set_title('20日滚动年化波动率对比', fontsize=14, fontweight='bold')
ax.set_ylabel('Volatility (%)')
ax.legend(); ax.grid(alpha=0.3)
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

print(f"A股平均波动率: {df_a['volatility'].mean():.1f}%")
print(f"港股平均波动率: {df_h['volatility'].mean():.1f}%")


In [ ]:
# 日收益率分布
fig, axes = plt.subplots(1, 2, figsize=(14,5))

axes[0].hist(df_a['ret'].dropna()*100, bins=50, color='#ef232a', alpha=0.7, edgecolor='white')
axes[0].axvline(0, color='#333', ls='--', lw=0.8)
axes[0].set_title(f'A股 日均收益 {df_a["ret"].mean()*100:.3f}%  Std {df_a["ret"].std()*100:.2f}%')
axes[0].set_xlabel('Daily Return (%)'); axes[0].set_ylabel('Frequency')

axes[1].hist(df_h['ret'].dropna()*100, bins=50, color='#f5a623', alpha=0.7, edgecolor='white')
axes[1].axvline(0, color='#333', ls='--', lw=0.8)
axes[1].set_title(f'港股 日均收益 {df_h["ret"].mean()*100:.3f}%  Std {df_h["ret"].std()*100:.2f}%')
axes[1].set_xlabel('Daily Return (%)'); axes[1].set_ylabel('Frequency')

plt.tight_layout(); plt.show()

print(f"A股日收益率标准差: {df_a['ret'].std()*100:.2f}%")
print(f"港股日收益率标准差: {df_h['ret'].std()*100:.2f}%")


## 分析结论

| 指标 | A股 (600276.SH) | 港股 (01276.HK) |
|------|:---:|:---:|
| 近一年涨跌幅 | +2.23% | +1.57% |
| 均价 | 59.18 元 | 71.01 HKD |
| 最高价 | 74.04 | 94.97 |
| 最低价 | 45.26 | 51.30 |

**数据来源**: Tushare Pro & westock-data
